# Spark Exercises 05 — Joins & Shuffles

Joins are the heart of every Foundry pipeline — and in Spark a join is also where the most expensive thing in distributed computing happens: a **shuffle**, where data flies across the network so matching keys end up on the same machine. This mission chapter covers every join type (including the SQL-flavoured `semi` / `anti`), how to read a join in `explain()`, and the **broadcast join** trick that avoids the shuffle for small tables.

**Data files:** `orders.csv`, `customers.csv`, `products.csv`, `regions.csv`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read all four files (`header=True`, `inferSchema=True`): `orders`, `customers`, `products`, `regions`.

### 3. **Inner join** `orders` to `customers` on `customer_id` (pass the key as a string so the column is not duplicated). Show `order_id`, `customer_name`, `segment`, `quantity` for 5 rows.

### 4. Confirm the inner join kept every order: compare `orders.join(customers, "customer_id").count()` to `orders.count()`.

### 5. **Left anti join** — find customers who have **no orders at all**. (`customers.join(orders, "customer_id", "left_anti")`). How many are there, and show 5.

### 6. **Left semi join** — keep only the customers who **do** have orders. Note a semi join returns **only the left table's columns** (it is a filter, not a real join). How many?

### 7. **Multi-table join.** Enrich `orders` with `customers`, then `regions` (on `region_id`), then `products` (on `product_sku`). Add `revenue = quantity * unit_price` and show `order_id, region_name, category, segment, revenue` for 5 rows.

### 8. Using the `full` table, compute **revenue per `region_name`** (positive quantities only), highest first.

### 9. **See the shuffle.** Call `.explain()` on `orders.join(customers, "customer_id")` and find the `Exchange` (that is the shuffle) and the `SortMergeJoin`.

### 10. **Broadcast join.** `regions` is tiny (5 rows), so broadcasting it avoids the shuffle entirely. Wrap it in `F.broadcast(...)`, join, and call `.explain()` — you should now see `BroadcastHashJoin` instead of `SortMergeJoin`.

### 11. **Watch out for ambiguous columns.** Both `orders` and a renamed `customers` share `customer_id`. Join on an explicit condition `orders.customer_id == customers.customer_id` and then try to select `"customer_id"`... it is ambiguous. The fix: join on the **string key** (as we did), or `drop` one side. Here, do it the safe way and select the customer's `region_id`.

### 12. Per `segment`, count **distinct customers who placed orders** and total revenue. Join orders+customers, then `groupBy('segment')` with `countDistinct('customer_id')` and `sum(quantity*unit_price)`.